# Practice: Python for data engineering

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python-09-practice-data-engineering.ipynb)

Companion to the [Python for data engineering](python-09-data-engineering.ipynb) chapter. The exercises build the small pieces of a pipeline: validation that keeps reasons, de-duplication, watermarks, partition paths, retry waits and reconciliation counts.

<details>
<summary><b>How to use this notebook</b></summary>

Work through the exercises in order. Each one has three parts:

1. **Question:** what to build, with the rules it must follow.
2. **Exercise cell:** replace the `pass` with your code. Keep the function name and parameters.
3. **Check cell:** run it after your code. Each line says `pass` when your function gives the expected result, and `todo` when it does not yet. Checks never stop the notebook, so you can run everything at any time.

The last cell prints your score. Use `Runtime -> Run all` to rerun everything after a change.

</details>

In [ ]:
_tally = {"passed": 0, "total": 0}

def check(label, make_actual, expected):
    """Run one check. It never raises, so the notebook always runs from top to bottom."""
    _tally["total"] += 1
    try:
        actual = make_actual()
    except Exception as error:
        print(f"error  {label}: {type(error).__name__}: {error}")
        return
    if actual == expected:
        _tally["passed"] += 1
        print(f"pass   {label}")
    else:
        print(f"todo   {label}: got {actual!r}")

def raises(make_actual, error_type):
    """True when the call raises error_type, False when it raises anything else or nothing."""
    try:
        make_actual()
    except error_type:
        return True
    except Exception:
        return False
    return False

def score():
    print(f"\nScore: {_tally['passed']} of {_tally['total']} checks pass")

## Exercise 1: validate a row and say why it fails

Write `validate_row(row)`. Return `(True, None)` for a valid row. Otherwise return `(False, reason)`, using exactly these reasons, checked in this order:

1. `"missing order_id"` when the `order_id` key is absent.
2. `"qty is not an integer"` when `qty` cannot be read as an `int`.
3. `"qty is negative"` when `qty` is below zero.

In [ ]:
def validate_row(row):
    # your code here
    pass

In [ ]:
check("valid row", lambda: validate_row({"order_id": 1, "qty": "3"}), (True, None))
check(
    "missing order_id",
    lambda: validate_row({"qty": "3"}),
    (False, "missing order_id"),
)
check(
    "qty is not an integer",
    lambda: validate_row({"order_id": 1, "qty": "x"}),
    (False, "qty is not an integer"),
)
check(
    "qty is negative",
    lambda: validate_row({"order_id": 1, "qty": "-2"}),
    (False, "qty is negative"),
)

<details>
<summary><b>Show solution</b></summary>

```python
def validate_row(row):
    if "order_id" not in row:
        return False, "missing order_id"
    try:
        qty = int(row.get("qty", ""))
    except ValueError:
        return False, "qty is not an integer"
    if qty < 0:
        return False, "qty is negative"
    return True, None
```

</details>

## Exercise 2: keep the last version of each record

Write `dedupe_by_key(rows, key)`. When several rows share the same value in `key`, keep the **last** one. Return the kept rows in the order their keys first appeared.

*Hint: a dictionary keeps the first insertion position but takes the latest value.*

In [ ]:
def dedupe_by_key(rows, key):
    # your code here
    pass

In [ ]:
check(
    "dedupe_by_key keeps the last row per key",
    lambda: dedupe_by_key(
        [{"id": 1, "v": "a"}, {"id": 2, "v": "b"}, {"id": 1, "v": "c"}],
        "id",
    ),
    [{"id": 1, "v": "c"}, {"id": 2, "v": "b"}],
)

<details>
<summary><b>Show solution</b></summary>

```python
def dedupe_by_key(rows, key):
    latest = {}
    for row in rows:
        latest[row[key]] = row
    return list(latest.values())
```

</details>

## Exercise 3: a watermark filter

Write `filter_newer(rows, watermark)`. Each row has a `ts` field holding an ISO date string. Keep the rows whose `ts` is **later** than `watermark`. Return a tuple `(kept_rows, new_watermark)`. `new_watermark` is the largest `ts` among the kept rows. If no row is kept, return the old watermark.

In [ ]:
def filter_newer(rows, watermark):
    # your code here
    pass

In [ ]:
check(
    "filter_newer keeps later rows and advances the watermark",
    lambda: filter_newer([{"ts": "2024-01-10"}, {"ts": "2024-01-12"}], "2024-01-10"),
    ([{"ts": "2024-01-12"}], "2024-01-12"),
)
check(
    "filter_newer keeps the old watermark when nothing is new",
    lambda: filter_newer([{"ts": "2024-01-10"}], "2024-02-01"),
    ([], "2024-02-01"),
)

<details>
<summary><b>Show solution</b></summary>

```python
def filter_newer(rows, watermark):
    kept = [row for row in rows if row["ts"] > watermark]
    if not kept:
        return kept, watermark
    return kept, max(row["ts"] for row in kept)
```

</details>

## Exercise 4: a Hive-style partition path

Write `partition_path(root, table, day)`. Return a `Path` for `<root>/<table>/dt=<day>/part-0000.jsonl`. Build it with `/`, not with string joins.

In [ ]:
from pathlib import Path

def partition_path(root, table, day):
    # your code here
    pass

In [ ]:
check(
    "partition_path(...)",
    lambda: partition_path("warehouse", "orders", "2024-01-15").as_posix(),
    "warehouse/orders/dt=2024-01-15/part-0000.jsonl",
)

<details>
<summary><b>Show solution</b></summary>

```python
from pathlib import Path

def partition_path(root, table, day):
    return Path(root) / table / f"dt={day}" / "part-0000.jsonl"
```

</details>

## Exercise 5: exponential retry waits

Write `retry_delays(attempts, base_delay)`. Return the list of waits between attempts, with no jitter. There is one wait between each pair of attempts, so `attempts - 1` waits in all. Each wait is double the one before it, starting at `base_delay`.

In [ ]:
def retry_delays(attempts, base_delay):
    # your code here
    pass

In [ ]:
check("retry_delays(4, 0.5)", lambda: retry_delays(4, 0.5), [0.5, 1.0, 2.0])
check("retry_delays(1, 1.0) == []", lambda: retry_delays(1, 1.0), [])

<details>
<summary><b>Show solution</b></summary>

```python
def retry_delays(attempts, base_delay):
    return [base_delay * 2 ** i for i in range(attempts - 1)]
```

</details>

## Exercise 6: count rejects by reason

Write `reason_counts(rejects)`. Each reject is a dictionary with a `reason` key. Return a plain `dict` that maps each reason to how many rejects had it.

*Hint: `collections.Counter` counts for you. Convert it with `dict(...)` before you return it.*

In [ ]:
def reason_counts(rejects):
    # your code here
    pass

In [ ]:
check(
    "reason_counts",
    lambda: reason_counts([{"reason": "a"}, {"reason": "b"}, {"reason": "a"}]),
    {"a": 2, "b": 1},
)
check("reason_counts([]) == {}", lambda: reason_counts([]), {})

<details>
<summary><b>Show solution</b></summary>

```python
from collections import Counter

def reason_counts(rejects):
    return dict(Counter(reject["reason"] for reject in rejects))
```

</details>

## Score

In [ ]:
score()